<a href="https://colab.research.google.com/github/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_6/lessons/lesson_52_final_project/note_lesson_52_final_project_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧭 Урок 52 — Робота над фінальним проєктом: аудит

Перед фінальним проєктом ми **міряємо** два проєкти курсу — `news_hub` (FastAPI) і проєкт нотаток (Django) — тими самими інструментами, якими потім перевіриш свій: граф імпортів, цикли, правило шарів, індекси, пагінація, кількість SQL-запитів, стан між репліками, чек-лист репозиторію. Усе працює на стандартній бібліотеці й SQLite, без Docker і без мережі (крім першого клонування в Colab).

| Розділ | Що всередині | Приклади | Задача |
|---|---|---|---|
| 0 | підготовка: корінь курсу, тимчасова папка | 0.1 | |
| 1 | **граф імпортів** через `ast`: що бачить розбір, відносні імпорти, CLI, fan-in / fan-out, гарячі точки, проєкт нотаток | 1.1–1.7 | |
| 2 | **цикли імпортів**: Тар'ян крок за кроком, зламане ребро, справжній `ImportError`, найкоротший шлях | 2.1–2.6 | |
| 3 | **правило шарів**: шари, глибина модуля, тест архітектури в pytest | 3.1–3.3 | **Задача 1** |
| 4 | **база**: індекс і вираз, `lower()` у SQLite, trigram-пошук, зайві індекси, ціна індексу на запис | 4.1–4.6 | |
| 5 | **пагінація**: OFFSET і keyset — швидкість, стабільність, курсор з двох полів | 5.1–5.3 | **Задача 2** |
| 6 | **N+1**: лічильник запитів на `sqlite3` і на SQLAlchemy | 6.1–6.3 | |
| 7 | **стан і репліки**: throttle і кеш у пам'яті процесу | 7.1–7.2 | |
| 8 | **чек-лист репозиторію**: Compose, `docker compose config`, залежності, секрети | 8.1–8.4 | **Задача 3** |

**Як працювати**

- Виконуй клітинки **зверху вниз**. Клітинка з визначенням (`def`, класи, константи, SQL) лише створює їх — **запуск завжди в наступній клітинці**.
- Перед **🔮 Прогнозом** спершу відповідай сам, потім розгортай відповідь.
- Час у мілісекундах у тебе буде інший — важливе **співвідношення** («у 100 разів швидше»), а не саме число.
- У книзі — справжні `EXPLAIN ANALYZE` на PostgreSQL з 302 400 новинами; тут — ті самі ідеї на SQLite, яка є в кожному Python. Повний розбір: [Урок 52](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m6/lesson_52/).

---
## 0. Підготовка

### Приклад 0.1. Де лежить курс

Інструменти уроку (`architecture_audit.py`) і проєкти, які ми аудитуємо (`module_5/lessons/lesson_51_ci_cd/`), — у репозиторії курсу. Функція `find_course_root` шукає його вгору від поточної папки (Jupyter у клонованому репозиторії), а якщо не знайшла (Colab) — клонує **лише потрібні папки** (`sparse-checkout`). Усе, що ноутбук створює (тестові пакети, копії проєктів), — у тимчасовій папці `WORK`, яку прибираємо наприкінці.

In [ ]:
import importlib.util
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

REPO = "https://github.com/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026.git"
LESSON = "module_6/lessons/lesson_52_final_project"
PROJECTS = "module_5/lessons/lesson_51_ci_cd"


def find_course_root(marker: str, sparse: list[str]) -> Path:
    """Корінь репозиторію курсу: шукаємо вгору від поточної папки; не знайшли (Colab) — клонуємо лише потрібні папки."""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / marker).is_dir():
            return folder
    target = here / "course_repo"
    if not target.is_dir():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", REPO, str(target)],
                       check=True)
        subprocess.run(["git", "-C", str(target), "sparse-checkout", "set", *sparse], check=True)
    return target


def ensure(package: str, module: str) -> None:
    """Встановити пакет, якщо його немає (у Colab і Jupyter курсу вони вже є)."""
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

In [ ]:
ensure("pyyaml", "yaml")
ensure("sqlalchemy", "sqlalchemy")

ROOT = find_course_root(LESSON, [LESSON, PROJECTS])
sys.path.insert(0, str(ROOT / LESSON))
from architecture_audit import collect, cycles

NEWS = ROOT / PROJECTS / "news_hub" / "news_hub"            # пакет news_hub (FastAPI)
NOTES = ROOT / PROJECTS / "crispy_notes_project"            # проєкт нотаток (Django)
WORK = Path(tempfile.mkdtemp(prefix="audit_"))              # усе тимчасове — сюди
print("папка уроку:", (ROOT / LESSON).is_dir(), "| news_hub:", NEWS.is_dir(), "| нотатки:", NOTES.is_dir())

---
## 1. Граф імпортів

`architecture_audit.py` читає кожен `.py` пакета через `ast` і збирає, хто кого імпортує. Проєкт не треба встановлювати: код **розбирається, а не виконується** — тому аудит працює навіть без FastAPI, SQLAlchemy чи бази.

### Приклад 1.1. Граф `news_hub`

`collect([пакет])` повертає два словники: `graph` — `{модуль: {внутрішні модулі, які він імпортує}}` і `lines` — кількість рядків у модулі. Чужі пакети (`sqlalchemy`, `fastapi`) у граф не потрапляють — лише модулі самого проєкту.

In [ ]:
graph, lines = collect([NEWS])
print(len(graph), "модулів,", sum(lines.values()), "рядків")
print("repository імпортує:", sorted(graph["news_hub.repository"]))
print("db імпортує:        ", sorted(graph["news_hub.db"]) or "нічого свого")

🔮 **Прогноз:** який модуль `news_hub` імпортує найбільше інших модулів проєкту?

<details>
<summary>Відповідь</summary>

`news_hub.api` — 18 модулів: кожен ендпоінт тягне свою частину системи. Це точка входу: її ніхто не імпортує, а вона — всіх. Перевіримо в прикладі 1.5.
</details>

### Приклад 1.2. Що бачить `ast`

Задача: зрозуміти, з чого `collect` будує ребра. Невелика зміна — розберемо кілька рядків імпорту вручну. `ast.Import` — це `import x`, `ast.ImportFrom` — `from x import y`; поле `level` — кількість крапок у відносному імпорті (`from ..models` → `level=2`).

In [ ]:
import ast

SAMPLE = """
import sqlalchemy
from .db import Base
from ..models import Note
from news_hub import cache


def later():
    from .repository import NewsRepository      # імпорт усередині функції
"""


def show_imports(source: str) -> None:
    """Надрукувати кожен імпорт так, як його бачить ast."""
    for node in ast.walk(ast.parse(source)):
        if isinstance(node, ast.Import):
            print(f"рядок {node.lineno}: Import      names={[a.name for a in node.names]}")
        elif isinstance(node, ast.ImportFrom):
            print(f"рядок {node.lineno}: ImportFrom  module={node.module!r:<14} level={node.level} "
                  f"names={[a.name for a in node.names]}")

In [ ]:
show_imports(SAMPLE)

Результат: `ast.walk` обходить **усе** дерево, тож бачить і імпорт усередині `later()` (рядок 9). Python виконає його лише під час виклику функції, але залежність від `repository` від цього не зникає — аудит її покаже. `level=1` і `level=2` — одна й дві крапки: `collect` відраховує стільки рівнів від пакета модуля.

**Правило:** аудит через `ast` дивиться на текст програми, а не на те, що встигло виконатися.

### Приклад 1.3. Відносні імпорти на крихітному пакеті

Задача: перевірити, що `from ..models import Note` з `app/sub/views.py` стає ребром до `app.models`. Створимо пакет у тимчасовій папці — так само роблять тести уроку (`test_architecture_audit.py`).

In [ ]:
def make_package(root: Path, files: dict[str, str]) -> Path:
    """Записати файли {'app/db.py': 'текст'} у root і повернути шлях до пакета (перша частина шляху)."""
    for name, source in files.items():
        path = root / name
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(source, encoding="utf-8")
    return root / next(iter(files)).split("/")[0]


APP_FILES = {
    "app/__init__.py": "",
    "app/db.py": "import sqlalchemy\n",                          # чужий пакет — не ребро
    "app/models.py": "from .db import Base\n",
    "app/api.py": "from app import models\nfrom app.db import Base\n",
    "app/sub/__init__.py": "",
    "app/sub/views.py": "from ..models import Note\nimport app.db\n",
}

In [ ]:
app = make_package(WORK / "demo", APP_FILES)
demo_graph, demo_lines = collect([app])
for name in sorted(demo_graph):
    print(f"{name:<14} → {sorted(demo_graph[name]) or '—'}")

`app.sub.views` залежить від `app.models` (дві крапки від `app/sub/` — це `app`) і від `app.db`. `from app import models` — ребро до **модуля** `app.models`, а не до пакета `app`. А `import sqlalchemy` ребра не дає: це не наш модуль.

### Приклад 1.4. Той самий аудит з командного рядка

Задача: отримати таблицю для README чи CI без Python-коду. Скрипт уроку друкує модулі з рядками, fan-in і fan-out, а з `--mermaid` — ще й текст діаграми, який GitHub малює в README.

In [ ]:
audit = subprocess.run([sys.executable, str(ROOT / LESSON / "architecture_audit.py"), str(NEWS), "--mermaid"],
                       capture_output=True, text=True, check=True)
out = audit.stdout.splitlines()
print("\n".join(out[:6]))
print("…")
print("\n".join(line for line in out if line.startswith("цикли")))
mermaid_lines = out[out.index("graph LR"):]
print(f"\nтекст діаграми: {len(mermaid_lines)} рядків, перші три:")
print("\n".join(mermaid_lines[:3]))

### Приклад 1.5. Fan-in і fan-out

- **fan-out** модуля — скільки модулів він імпортує (`len(graph[m])`);
- **fan-in** — скільки модулів імпортують його.

Високий fan-in і нульовий fan-out — **фундамент**: від нього залежать усі, він — ні від кого. Нульовий fan-in — **точка входу**: її запускають, а не імпортують. Порахуємо fan-in, обійшовши всі ребра один раз.

In [ ]:
def fan_in(graph: dict[str, set[str]]) -> dict[str, int]:
    """{модуль: скільки модулів його імпортують} — для кожного модуля графа, навіть з нулем."""
    counts = {name: 0 for name in graph}
    for targets in graph.values():
        for target in targets:
            counts[target] += 1
    return counts

In [ ]:
incoming = fan_in(graph)
for name in sorted(graph, key=lambda m: -incoming[m])[:6]:
    print(f"{name:<22} fan-in {incoming[name]}  fan-out {len(graph[name])}")
foundation = {m for m in graph if incoming[m] >= 4 and not graph[m]}
print("фундамент:", sorted(foundation))
assert incoming["news_hub.llm"] == 7 and incoming["news_hub.api"] == 0
assert foundation == {"news_hub.llm", "news_hub.cache", "news_hub.db", "news_hub.parser"}

`llm` імпортують 7 модулів, а сам він не імпортує нічого свого. Інтерфейс такого модуля змінювати найдорожче — зміна зачепить усіх сімох, тож і тестів у нього має бути найбільше.

### Приклад 1.6. Точки входу й гарячі точки

Задача: знайти, з чого почати рефакторинг. Кандидат — великий модуль з великим fan-out: його важко читати, і він ламається від змін у багатьох місцях. Невелика зміна до прикладу 1.5 — сортуємо за `рядки × fan-out`.

In [ ]:
def entry_points(graph: dict[str, set[str]], incoming: dict[str, int]) -> list[str]:
    """Модулі, які ніхто не імпортує, але які самі щось імпортують (кореневі __init__ без коду — ні)."""
    return sorted(m for m in graph if incoming[m] == 0 and graph[m])


def hotspots(graph: dict[str, set[str]], lines: dict[str, int], top: int = 3) -> list[tuple[str, int, int]]:
    """(модуль, рядків, fan-out) — найбільші за добутком рядків на fan-out."""
    rows = [(m, lines[m], len(graph[m])) for m in graph]
    return sorted(rows, key=lambda row: -row[1] * row[2])[:top]

In [ ]:
print("точки входу:", entry_points(graph, incoming))
for name, size, out_degree in hotspots(graph, lines):
    print(f"{name:<22} {size:>4} рядків  fan-out {out_degree:>2}")

`api.py` — 770 рядків і 18 залежностей: книга пропонує розкласти його на `APIRouter` за тегами (news, admin, analyze, …). Шляхи й тести від цього не змінюються, а кожен файл стає вдесятеро меншим.

### Приклад 1.7. Той самий аудит для Django

Задача: перевірити, що інструмент працює не лише для FastAPI. Проєкт нотаток — два пакети: застосунок `hello_app` і налаштування `hello_project`. `collect` приймає список пакетів і пропускає `migrations` і `tests*.py`.

In [ ]:
notes_graph, notes_lines = collect([NOTES / "hello_app", NOTES / "hello_project"])
notes_in = fan_in(notes_graph)
print(len(notes_graph), "модулів,", sum(notes_lines.values()), "рядків; цикли:", cycles(notes_graph) or "немає")
for name in sorted(notes_graph, key=lambda m: (-notes_in[m], m))[:5]:
    print(f"{name:<28} fan-in {notes_in[name]}  fan-out {len(notes_graph[name])}")

Фундамент нотаток — `models`, а одразу за ним `selectors` і `services`: їх використовують і сторінки, і API, і WebSocket-чат. Саме тому правило «хто бачить нотатку» написане один раз (урок 45).

---
## 2. Цикли імпортів

Цикл — модулі, які через ланцюжок імпортів залежать самі від себе: `A → B → C → A`. Python такий пакет часто не імпортує (`ImportError: … partially initialized module`), а якщо й імпортує — лише доки ніхто не змінить порядок рядків.

`cycles(graph)` шукає **сильно зв'язані компоненти** алгоритмом Тар'яна: один обхід у глибину, і кожен модуль отримує два числа:

- `index` — порядковий номер, коли обхід уперше до нього дійшов;
- `low` — найменший `index`, до якого можна «повернутися» з нього по ребрах у модулі, що ще на стеку.

Коли обхід виходить з модуля, у якого `low == index`, — це корінь компоненти: знімаємо зі стеку все до нього включно. Понад один модуль — цикл.

```text
orders → users → payments → orders        utils → orders

крок 1  orders    index 0, low 0    стек: orders
крок 2  users     index 1, low 1    стек: orders, users
крок 3  payments  index 2, low 2    стек: orders, users, payments
крок 4  payments → orders: orders ще на стеку → low[payments] = 0
крок 5  назад: low[users] = min(1, 0) = 0;  low[orders] = 0 = index[orders]
крок 6  orders — корінь: знімаємо payments, users, orders → цикл із 3 модулів
крок 7  utils     index 3, low 3;  utils → orders: вже не на стеку → компонента з 1 модуля, не цикл
```

### Приклад 2.1. Цикл у магазині

In [ ]:
SHOP = {
    "orders": {"users"},
    "users": {"payments"},
    "payments": {"orders"},
    "utils": {"orders"},
}

In [ ]:
print("цикли в SHOP:", cycles(SHOP))
print("цикли в news_hub:", cycles(graph) or "немає")

`utils` залежить від циклу, але сам у ньому не бере участі: з `orders` до `utils` дороги немає.

### Приклад 2.2. Тар'ян з друком кожного кроку

Задача: побачити таблицю вище не на папері, а у виконанні. Невелика зміна до `cycles` — та сама логіка, але кожен крок друкується з відступом за глибиною обходу.

In [ ]:
def tarjan_trace(graph: dict[str, set[str]]) -> None:
    """Алгоритм Тар'яна, що друкує index, low і стек на кожному кроці."""
    index: dict[str, int] = {}
    low: dict[str, int] = {}
    stack: list[str] = []
    on_stack: set[str] = set()

    def visit(node: str, depth: int) -> None:
        pad = "    " * depth
        index[node] = low[node] = len(index)
        stack.append(node)
        on_stack.add(node)
        print(f"{pad}→ {node}: index {index[node]}, стек {stack}")
        for nxt in sorted(graph.get(node, ())):
            if nxt not in index:
                visit(nxt, depth + 1)
                low[node] = min(low[node], low[nxt])
                print(f"{pad}  ← повернулись з {nxt}: low[{node}] = {low[node]}")
            elif nxt in on_stack:
                low[node] = min(low[node], index[nxt])
                print(f"{pad}  {node} → {nxt}: {nxt} ще на стеку, low[{node}] = {low[node]}")
        if low[node] == index[node]:
            component = []
            while True:
                member = stack.pop()
                on_stack.discard(member)
                component.append(member)
                if member == node:
                    break
            verdict = "ЦИКЛ" if len(component) > 1 else "не цикл"
            print(f"{pad}✔ {node} — корінь компоненти {sorted(component)}: {verdict}")

    for node in sorted(graph):
        if node not in index:
            visit(node, 0)

In [ ]:
tarjan_trace(SHOP)

Вивід повторює таблицю: `payments` бачить `orders` на стеку й опускає свій `low` до 0, це «спливає» до `users` і `orders`, і на `orders` компонента закривається. `utils` обходиться останнім і стає окремою компонентою з одного модуля.

### Приклад 2.3. Одне зайве ребро в `news_hub`

Задача: що буде, якщо в `db.py` з'явиться рядок `from .repository import NewsRepository`? Невелика зміна — додаємо одне ребро в **копію** графа (сам `graph` не чіпаємо).

🔮 **Прогноз:** скільки модулів буде в циклі — два (`db`, `repository`) чи більше?

In [ ]:
broken = {name: set(targets) for name, targets in graph.items()}     # копія: graph не чіпаємо
broken["news_hub.db"].add("news_hub.repository")

In [ ]:
found = cycles(broken)
print(found)
assert found == [["news_hub.db", "news_hub.repository", "news_hub.tables"]]
assert cycles(graph) == []

<details>
<summary>Відповідь</summary>

Три: `repository` імпортує `tables`, а `tables` — `db`. Тож з `db` можна дійти до `repository` і назад, і `tables` на цьому шляху — теж частина циклу.
</details>

### Приклад 2.4. Чому саме `tables`: найкоротший шлях

Задача: пояснити цикл людині, яка його не бачить. Найкоротший шлях між двома модулями шукає пошук у ширину (BFS): спершу всі сусіди, потім сусіди сусідів.

In [ ]:
from collections import deque


def shortest_path(graph: dict[str, set[str]], start: str, goal: str) -> list[str]:
    """Найкоротший ланцюжок імпортів від start до goal (пошук у ширину) або []."""
    previous = {start: None}
    queue = deque([start])
    while queue:
        node = queue.popleft()
        if node == goal:
            path = []
            while node is not None:
                path.append(node)
                node = previous[node]
            return path[::-1]
        for nxt in sorted(graph.get(node, ())):
            if nxt not in previous:
                previous[nxt] = node
                queue.append(nxt)
    return []

In [ ]:
print("db → tables:    ", " → ".join(shortest_path(broken, "news_hub.db", "news_hub.tables")))
print("tables → db:    ", " → ".join(shortest_path(broken, "news_hub.tables", "news_hub.db")))
print("без нового ребра db → tables:", shortest_path(graph, "news_hub.db", "news_hub.tables") or "дороги немає")

Модуль належить циклу, якщо з нього можна дійти до іншого модуля циклу **і повернутися**. `tables → db` було й раніше, а нове ребро відкрило дорогу назад: `db → repository → tables`. Разом — коло `db → repository → tables → db`. Без нового ребра з `db` до `tables` дороги немає. Ці два рядки й варто вставити в опис PR, коли просиш колегу прибрати імпорт.

### Приклад 2.5. Справжній `ImportError`

Задача: побачити, як цикл ламає запуск. Два модулі, що імпортують імена один з одного, — і `python -c "import shop.orders"` в окремому процесі.

In [ ]:
CYCLE_FILES = {
    "shop/__init__.py": "",
    "shop/orders.py": "from shop.users import User\n\n\nclass Order:\n    pass\n",
    "shop/users.py": "from shop.orders import Order\n\n\nclass User:\n    pass\n",
}
LAZY_USERS = "class User:\n    def orders(self):\n        from shop.orders import Order      # відкладений імпорт\n        return [Order()]\n"


def try_import(root: Path, module: str) -> str:
    """Імпортувати модуль в окремому процесі; повернути 'OK' або останній рядок помилки."""
    run = subprocess.run([sys.executable, "-c", f"import {module}"], cwd=root, capture_output=True, text=True)
    return "OK" if run.returncode == 0 else run.stderr.strip().splitlines()[-1]

In [ ]:
cyc = make_package(WORK / "cycle", CYCLE_FILES)
print("цикл на рівні модулів:", try_import(cyc.parent, "shop.orders"))

(cyc / "users.py").write_text(LAZY_USERS, encoding="utf-8")         # імпорт переносимо всередину методу
print("відкладений імпорт:   ", try_import(cyc.parent, "shop.orders"))
print("а аудит усе одно бачить:", cycles(collect([cyc])[0]))

Перший запуск падає з `partially initialized module`: `orders` почав виконуватися, імпортує `users`, а той просить `Order` з `orders`, який ще не дійшов до рядка `class Order`. Відкладений імпорт у методі «лікує» запуск, але не архітектуру: два модулі досі залежать один від одного, і аудит це показує. Правильне виправлення — винести спільне в третій модуль, від якого залежать обидва.

### Приклад 2.6. Цикл, якого не видно

🔮 **Прогноз:** у `news_hub` циклів немає. Але чи означає це, що будь-яке нове ребро між двома **фундаментальними** модулями безпечне? Додамо `cache → llm` (обидва з fan-out 0).

<details>
<summary>Відповідь</summary>

Циклу не буде — у `llm` немає жодного свого імпорту, з нього не повернутись до `cache`. Але `cache` перестає бути фундаментом: тепер кожна зміна в `llm` зачіпає й усіх, хто імпортує `cache`. Цикли — найгрубіша помилка; тонші ловить правило шарів (розділ 3).
</details>

In [ ]:
extra = {name: set(targets) for name, targets in graph.items()}
extra["news_hub.cache"].add("news_hub.llm")

In [ ]:
print("цикли:", cycles(extra) or "немає")
print("fan-out cache:", len(graph["news_hub.cache"]), "→", len(extra["news_hub.cache"]))

---
## 3. Правило шарів

Цикли ловлять, коли залежності пішли по колу. Але архітектуру ламає й одне ребро «не в той бік»: фундамент, що імпортує точку входу. Домовимось про три шари `news_hub`:

```text
2 · точки входу   api, jobs, bot, bot.handlers, bot.middlewares …
        │  можна імпортувати лише вниз
        ▼
1 · логіка        repository, analysis, notify, scraper, rss, middleware
        │
        ▼
0 · фундамент     db, tables, models, parser, llm, cache, security …
```

Правило: модуль може імпортувати лише **свій шар або нижчий**.

### Приклад 3.1. Шари як дані

In [ ]:
ENTRY = {"news_hub", "news_hub.api", "news_hub.jobs", "news_hub.bot", "news_hub.bot.__main__",
         "news_hub.bot.factory", "news_hub.bot.handlers", "news_hub.bot.middlewares"}
LOGIC = {"news_hub.repository", "news_hub.analysis", "news_hub.notify", "news_hub.scraper", "news_hub.rss",
         "news_hub.middleware"}
layer = {name: 2 if name in ENTRY else 1 if name in LOGIC else 0 for name in graph}

In [ ]:
for level, title in ((2, "точки входу"), (1, "логіка"), (0, "фундамент")):
    names = sorted(n.removeprefix("news_hub.") for n in layer if layer[n] == level)
    print(f"{level} · {title:<12} {len(names):>2}: {', '.join(names)}")

### Приклад 3.2. Глибина модуля: шари з самого графа

Задача: як призначити шари, коли модулів сотня? Невелика зміна — рахуємо **глибину**: 0 для модуля без своїх імпортів, інакше 1 + найбільша глибина тих, кого він імпортує. Працює лише для графа без циклів — і це ще одна причина їх не мати.

In [ ]:
from functools import lru_cache


def depth(graph: dict[str, set[str]]) -> dict[str, int]:
    """Найдовший ланцюжок імпортів униз від кожного модуля (граф без циклів)."""
    @lru_cache(maxsize=None)
    def d(module: str) -> int:
        return 1 + max((d(target) for target in graph[module]), default=-1)

    return {module: d(module) for module in graph}

In [ ]:
levels = depth(graph)
for name in sorted(levels, key=lambda m: (-levels[m], m))[:6]:
    print(f"глибина {levels[name]}  {name:<24} домовлений шар {layer[name]}")
print("глибина repository:", levels["news_hub.repository"], "| глибина 0:", sum(1 for v in levels.values() if v == 0), "модулів")

Глибина — підказка, а не закон: `api` найглибший, бо тягне все, а `repository` (шар 1) на глибині 2 — бо стоїть на `tables`, що стоїть на `db`. Шари — **рішення команди**, глибина лише показує, чи граф з ним згоден.

### Задача 1. Ребра «не в той бік»

Напиши `violations(graph, layer)` → множину ребер `(звідки, куди)`, де `layer[куди] > layer[звідки]` (модуль нижчого шару імпортує вищий). Перевірка — у наступній клітинці: у справжньому `news_hub` порушень немає, а «ярлик» `cache → api` («швидко взяти settings з api») — одне.

In [ ]:
def violations(graph: dict[str, set[str]], layer: dict[str, int]) -> set[tuple[str, str]]:
    # YOUR CODE HERE
    pass


In [ ]:
shortcut = {name: set(targets) for name, targets in graph.items()}
shortcut["news_hub.cache"].add("news_hub.api")                      # «швидко взяти settings з api»
print("порушення в news_hub:", violations(graph, layer) or "немає")
print("з ярликом:", violations(shortcut, layer))
assert isinstance(violations(graph, layer), set), "поверни множину пар (звідки, куди)"
assert violations(graph, layer) == set(), "у справжньому news_hub порушень немає — перевір напрям порівняння"
assert violations(shortcut, layer) == {("news_hub.cache", "news_hub.api")}
print("✅ Задача 1 розв'язана")

Так само влаштований `tests_architecture.py` проєкту нотаток (урок 45): тест розбирає код через `ast` і падає, якщо правило порушено.

### Приклад 3.3. Тест архітектури в pytest

Задача: щоб правило не жило лише в голові, воно має падати в CI. Невелика зміна — запишемо тест у файл і запустимо `pytest` двічі: на справжньому `news_hub` і на його копії, де в `db.py` додано зайвий імпорт. Тест не залежить від задачі 1: правило «фундамент нічого свого не імпортує» записане в ньому самому.

In [ ]:
ARCH_TEST = """
import os
import sys
from pathlib import Path

sys.path.insert(0, os.environ["AUDIT_DIR"])
from architecture_audit import collect, cycles

PACKAGE = Path(os.environ["PACKAGE"])
FOUNDATION = {"news_hub.db", "news_hub.cache", "news_hub.llm", "news_hub.parser"}


def test_no_import_cycles():
    graph, _ = collect([PACKAGE])
    assert cycles(graph) == []


def test_foundation_has_no_internal_imports():
    graph, _ = collect([PACKAGE])
    assert {m: sorted(graph[m]) for m in FOUNDATION if graph[m]} == {}
"""


def run_arch_test(package: Path) -> str:
    """Записати тест у WORK і запустити pytest для пакета; повернути підсумковий рядок."""
    test_file = WORK / "test_arch.py"
    test_file.write_text(ARCH_TEST, encoding="utf-8")
    env = {**os.environ, "AUDIT_DIR": str(ROOT / LESSON), "PACKAGE": str(package)}
    run = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "--color=no", str(test_file)],
                         capture_output=True, text=True, env=env, cwd=WORK)
    failed = [line[:110] for line in run.stdout.splitlines() if line.startswith("FAILED")]
    return "\n".join(failed + [run.stdout.strip().splitlines()[-1]])

In [ ]:
if importlib.util.find_spec("pytest") is None:
    print("pytest не встановлений: pip install pytest")
else:
    print("справжній news_hub:\n" + run_arch_test(NEWS))
    copy = WORK / "news_hub_copy" / "news_hub"
    shutil.copytree(NEWS, copy, dirs_exist_ok=True)
    db_file = copy / "db.py"
    db_file.write_text("from .repository import NewsRepository\n" + db_file.read_text(encoding="utf-8"), encoding="utf-8")
    print("\nкопія з імпортом у db.py:\n" + run_arch_test(copy))

На справжньому пакеті — `2 passed`; на копії обидва тести падають, і `pytest` показує, що саме не так. Такий тест у `tests/` фінального проєкту коштує 15 рядків і тримає архітектуру, коли код росте.

---
## 4. База: індекс і вираз

У книзі — головна знахідка аудиту бази `news_hub`: індекс trigram на `title` існував, займав 59 MB і **не використовувався**, бо запит `search()` порівнює `lower(title)`. Індекс допомагає лише виразу, під який побудований. Те саме видно на SQLite з B-tree індексом.

### Приклад 4.1. Індекс на `lower(category)` і запит без `lower`

Таблиця на 200 000 новин. Категорії — латиницею з великої літери (`Sport`), чому — у прикладі 4.2. `EXPLAIN QUERY PLAN` показує план: `SCAN` — перегляд усієї таблиці, `SEARCH … USING INDEX` — пошук за індексом.

In [ ]:
import random
import sqlite3
import time

CATEGORIES = ["News", "Economy", "Sport", "Tech", "Culture"]
NEWS_SCHEMA = """
CREATE TABLE news (id INTEGER PRIMARY KEY, title TEXT NOT NULL, category TEXT NOT NULL);
CREATE INDEX ix_news_category_lower ON news (lower(category));
"""


def plan(conn: sqlite3.Connection, sql: str, *params) -> str:
    """План запиту SQLite одним рядком."""
    return " | ".join(row[-1] for row in conn.execute("EXPLAIN QUERY PLAN " + sql, params))


def timed(conn: sqlite3.Connection, sql: str, *params, repeat: int = 20) -> float:
    """Середній час запиту в мілісекундах."""
    start = time.perf_counter()
    for _ in range(repeat):
        conn.execute(sql, params).fetchall()
    return (time.perf_counter() - start) / repeat * 1000

In [ ]:
random.seed(51)
conn = sqlite3.connect(":memory:")
conn.executescript(NEWS_SCHEMA)
conn.executemany("INSERT INTO news (title, category) VALUES (?, ?)",
                 ((f"Новина {i}", random.choice(CATEGORIES)) for i in range(200_000)))
print("category = ?        →", plan(conn, "SELECT id FROM news WHERE category = ?", "Sport"))
print("lower(category) = ? →", plan(conn, "SELECT id FROM news WHERE lower(category) = ?", "sport"))
slow = timed(conn, "SELECT id FROM news WHERE category = ?", "Sport")
fast = timed(conn, "SELECT id FROM news WHERE lower(category) = ?", "sport")
print(f"перегляд {slow:.1f} мс, індекс {fast:.1f} мс")

🔮 **Прогноз:** індекс є на `lower(category)`. Чому перший запит усе одно `SCAN`?

<details>
<summary>Відповідь</summary>

Для бази `category` і `lower(category)` — різні вирази. Значення в індексі вже перетворені `lower()`, і шукати в ньому «Sport» як є не можна. Запит і індекс мають збігатися дослівно: або індекс на `category`, або запит з `lower(category)`.
</details>

Обидва запити повертають ту саму п'яту частину таблиці (~40 000 рядків), тому різниця в часі тут помірна: багато часу йде на те, щоб віддати рядки. Найбільше індекс виграє, коли відповідь маленька, — як у пошуку в прикладі 4.3.

### Приклад 4.2. Пастка SQLite: `lower()` і кирилиця

Задача: чому категорії латиницею? Невелика зміна — та сама функція `lower()` на українському слові. PostgreSQL перетворює регістр за правилами мови; вбудовані `lower()` і `LIKE` у SQLite — **лише для латиниці**.

In [ ]:
probe = sqlite3.connect(":memory:")
print("lower('Sport') =", probe.execute("SELECT lower('Sport')").fetchone()[0])
print("lower('Спорт') =", probe.execute("SELECT lower('Спорт')").fetchone()[0])
print("'Спорт' LIKE 'спорт' →", probe.execute("SELECT 'Спорт' LIKE 'спорт'").fetchone()[0])
print("Python: 'Спорт'.lower() =", "Спорт".lower())
probe.close()

`lower('Спорт')` у SQLite лишився `Спорт`, а `LIKE` не вважає «Спорт» і «спорт» однаковими. Тести на SQLite можуть пройти там, де PostgreSQL поводиться інакше (і навпаки) — тому CI курсу ганяє тести `news_hub` ще й на PostgreSQL (урок 51).

**Правило:** пошук без урахування регістру для кирилиці перевіряй на тій базі, що в продакшені.

### Приклад 4.3. Пошук підрядка: перегляд і trigram

`LIKE '%кременчук%'` — `%` на початку означає «будь-де», і звичайний B-tree тут безсилий. PostgreSQL для цього має `pg_trgm` (книга: 372 мс → 0,14 мс). У SQLite є той самий принцип — повнотекстова таблиця FTS5 з токенізатором `trigram`: рядок розбирається на трійки символів («кре», «рем», «еме», …), і пошук іде за ними.

In [ ]:
WORDS = ["Київ", "Львів", "Одеса", "Харків", "Дніпро", "енергетика", "тариф", "погода", "футбол", "бюджет"]
FTS_SCHEMA = "CREATE VIRTUAL TABLE news_fts USING fts5(title, tokenize='trigram')"


def headline(i: int) -> str:
    """Заголовок з двох випадкових слів; «Кременчук» — лише в трьох новинах."""
    city = "Кременчук" if i in (7, 77_777, 199_999) else random.choice(WORDS)
    return f"{city}: {random.choice(WORDS)}, новина {i}"

In [ ]:
random.seed(52)
search = sqlite3.connect(":memory:")
search.execute("CREATE TABLE news (id INTEGER PRIMARY KEY, title TEXT NOT NULL)")
search.executemany("INSERT INTO news (id, title) VALUES (?, ?)", ((i, headline(i)) for i in range(200_000)))
try:
    search.execute(FTS_SCHEMA)
    search.execute("INSERT INTO news_fts (rowid, title) SELECT id, title FROM news")
    like_sql = "SELECT id FROM news WHERE title LIKE ?"
    fts_sql = "SELECT rowid FROM news_fts WHERE news_fts MATCH ?"
    print("LIKE:", plan(search, like_sql, "%Кременчук%"))
    print("FTS: ", plan(search, fts_sql, '"кременчук"'))
    print("знайдено:", [r[0] for r in search.execute(like_sql, ("%Кременчук%",))],
          [r[0] for r in search.execute(fts_sql, ('"кременчук"',))])
    slow, fast = timed(search, like_sql, "%Кременчук%", repeat=5), timed(search, fts_sql, '"кременчук"', repeat=5)
    print(f"LIKE {slow:.1f} мс, trigram {fast:.3f} мс — у {slow / fast:.0f} разів швидше")
except sqlite3.OperationalError as error:                       # SQLite старіша за 3.34 — без trigram
    print("trigram недоступний у цій SQLite:", error)

`LIKE` переглядає всі 200 000 заголовків, trigram-таблиця знаходить ті самі три новини за частки мілісекунди. Зверни увагу: `LIKE` шукав «Кременчук» з великої (бо `LIKE` у SQLite не знає регістру кирилиці), а trigram знайшов «кременчук» з малої — його токенізатор порівнює без регістру. Ціна — ще одна таблиця й час на її наповнення, як і 59 MB індексу `pg_trgm` у книзі.

### Приклад 4.4. Зайві індекси

`subscriptions` з `news_hub`: `UNIQUE (chat_id, keyword)` і окремий індекс на `chat_id`. Індекс зайвий, якщо його стовпці — **початок** стовпців іншого індексу тієї ж таблиці (як телефонна книга за «прізвище, ім'я» вже впорядкована за прізвищем). SQLite віддає список індексів через `PRAGMA index_list(таблиця)` (друге поле — назва), а стовпці індексу — через `PRAGMA index_info(назва)` (третє поле — ім'я стовпця).

In [ ]:
SUBSCRIPTIONS_SCHEMA = """
CREATE TABLE subscriptions (id INTEGER PRIMARY KEY, chat_id INTEGER NOT NULL, keyword TEXT NOT NULL,
                            UNIQUE (chat_id, keyword));
CREATE INDEX ix_subscriptions_chat_id ON subscriptions (chat_id);
CREATE INDEX ix_subscriptions_keyword ON subscriptions (keyword);
"""


def columns(conn: sqlite3.Connection, index: str) -> tuple[str, ...]:
    """Стовпці індексу в порядку, в якому він за ними впорядкований."""
    return tuple(row[2] for row in conn.execute(f"PRAGMA index_info('{index}')"))


def redundant(conn: sqlite3.Connection, table: str) -> set[tuple[str, str]]:
    """Пари (зайвий індекс, той, що його покриває): стовпці першого — початок стовпців другого."""
    indexes = {row[1]: columns(conn, row[1]) for row in conn.execute(f"PRAGMA index_list('{table}')")}
    return {(small, big) for small, cols in indexes.items() for big, other in indexes.items()
            if small != big and len(cols) < len(other) and other[:len(cols)] == cols}

In [ ]:
conn.executescript(SUBSCRIPTIONS_SCHEMA)
for row in conn.execute("PRAGMA index_list('subscriptions')"):
    print(f"{row[1]:<36} {columns(conn, row[1])}")
print("зайві:", redundant(conn, "subscriptions"))
assert redundant(conn, "subscriptions") == {("ix_subscriptions_chat_id", "sqlite_autoindex_subscriptions_1")}
assert redundant(conn, "news") == set()

`ix_subscriptions_keyword` не зайвий: `keyword` — **другий** стовпець унікального індексу, а за другим стовпцем книга не впорядкована. На PostgreSQL те саме робить `db_audit.sql` уроку — і ще порівнює класи операторів (Django-індекси `*_like` не дублі).

### Приклад 4.5. Перевірка: прибрали індекс — план не погіршився

Задача: довести, що індекс зайвий, **виміром**, а не міркуванням. Невелика зміна — план запиту `for_chat()` до і після `DROP INDEX`.

In [ ]:
FOR_CHAT = "SELECT keyword FROM subscriptions WHERE chat_id = ? ORDER BY keyword"

In [ ]:
conn.executemany("INSERT INTO subscriptions (chat_id, keyword) VALUES (?, ?)",
                 ((1_000_000 + i % 5000, f"слово{i}") for i in range(50_000)))
print("з ix_subscriptions_chat_id:", plan(conn, FOR_CHAT, 1_000_155))
conn.execute("DROP INDEX ix_subscriptions_chat_id")
print("без нього:                 ", plan(conn, FOR_CHAT, 1_000_155))
print("ключові слова чату:", [row[0] for row in conn.execute(FOR_CHAT, (1_000_155,))])

План не змінився: SQLite **ще до видалення** обирав унікальний `(chat_id, keyword)` як **покривний** (`COVERING INDEX`): у ньому є і умова пошуку, і готовий порядок `ORDER BY keyword`, і самі значення — таблицю читати не треба. Окремий `ix_subscriptions_chat_id` лише оновлювався на кожен запис. На PostgreSQL у книзі те саме: `Index Only Scan using uq_subscription_chat_keyword`.

### Приклад 4.6. Ціна індексу — на запис

Задача: «зайвий індекс нічого не коштує, просто лежить» — так? Невелика зміна — вставимо однакові 100 000 рядків у таблицю без індексів, з одним і з трьома.

In [ ]:
def insert_ms(extra_indexes: int, rows: int = 100_000) -> float:
    """Час вставки rows рядків у таблицю з extra_indexes додатковими індексами, мс."""
    db = sqlite3.connect(":memory:")
    db.execute("CREATE TABLE t (id INTEGER PRIMARY KEY, a INTEGER, b TEXT, c TEXT)")
    for i, column in enumerate(["a", "b", "c"][:extra_indexes]):
        db.execute(f"CREATE INDEX ix_{i} ON t ({column})")
    data = [(i * 7919 % rows, f"b{i}", f"c{i % 977}") for i in range(rows)]
    start = time.perf_counter()
    db.executemany("INSERT INTO t (a, b, c) VALUES (?, ?, ?)", data)
    db.commit()
    db.close()
    return (time.perf_counter() - start) * 1000

In [ ]:
timings = {count: insert_ms(count) for count in (0, 1, 3)}
for count, ms in timings.items():
    print(f"індексів {count}: {ms:6.0f} мс  ×{ms / timings[0]:.1f}")

Кожен індекс — ще одна впорядкована структура, яку база оновлює на **кожен** `INSERT`, `UPDATE` і `DELETE`. Тому правило аудиту двобічне: індекс під справжній запит — так; індекс «про всяк випадок» чи дубль — прибрати.

---
## 5. Пагінація: OFFSET і keyset

`GET /api/news?skip=…` робить `ORDER BY id OFFSET skip LIMIT 50`. Щоб віддати глибоку сторінку, база читає й викидає всі попередні рядки. Keyset-пагінація просить «наступні після `id = X`» — і починає читати одразу з потрібного місця індексу.

```text
запит 1: after = 0   WHERE id > 0 LIMIT 3   → id 1, 2, 3        next_after = 3
запит 2: after = 3   WHERE id > 3 LIMIT 3   → id 4, 5, 7 (6 видалено)   next_after = 7
запит 3: after = 7   WHERE id > 7 LIMIT 3   → id 8               менше за LIMIT → кінець
```

### Приклад 5.1. Глибока сторінка

In [ ]:
deep_offset = timed(conn, "SELECT id, title FROM news ORDER BY id LIMIT 50 OFFSET ?", 190_000)
deep_keyset = timed(conn, "SELECT id, title FROM news WHERE id > ? ORDER BY id LIMIT 50", 190_000)
print(f"OFFSET 190000: {deep_offset:.2f} мс")
print(f"id > 190000:   {deep_keyset:.3f} мс  (у {deep_offset / deep_keyset:.0f} разів швидше)")
print(plan(conn, "SELECT id FROM news WHERE id > ? ORDER BY id LIMIT 50", 190_000))
assert deep_keyset * 10 < deep_offset

`SEARCH news USING INTEGER PRIMARY KEY (rowid>?)` — база одразу стає на потрібне місце. `OFFSET` мусить пройти 190 000 рядків, щоб їх викинути; на PostgreSQL у книзі — 264 мс проти 0,08 мс.

### Приклад 5.2. Стабільність: між запитами видалили новину

Задача: друга перевага keyset — не швидкість. Невелика зміна — між першою і другою сторінкою видаляємо новину з першої сторінки.

🔮 **Прогноз:** яку новину клієнт з `OFFSET` не побачить ніколи?

In [ ]:
def offset_page(db: sqlite3.Connection, page: int, size: int) -> list[int]:
    return [r[0] for r in db.execute("SELECT id FROM feed ORDER BY id LIMIT ? OFFSET ?", (size, page * size))]


def keyset_page(db: sqlite3.Connection, after: int, size: int) -> list[int]:
    return [r[0] for r in db.execute("SELECT id FROM feed WHERE id > ? ORDER BY id LIMIT ?", (after, size))]


def fresh_feed() -> sqlite3.Connection:
    db = sqlite3.connect(":memory:")
    db.execute("CREATE TABLE feed (id INTEGER PRIMARY KEY)")
    db.executemany("INSERT INTO feed (id) VALUES (?)", ((i,) for i in range(1, 10)))
    return db

In [ ]:
for name in ("OFFSET", "keyset"):
    feed = fresh_feed()
    first = offset_page(feed, 0, 3) if name == "OFFSET" else keyset_page(feed, 0, 3)
    feed.execute("DELETE FROM feed WHERE id = 2")                     # редактор зняв новину
    second = offset_page(feed, 1, 3) if name == "OFFSET" else keyset_page(feed, first[-1], 3)
    print(f"{name:<7} сторінка 1: {first}  сторінка 2: {second}")

<details>
<summary>Відповідь</summary>

Новину з `id = 4`. Після видалення все зсунулось на одну позицію, `OFFSET 3` пропустив перші три рядки — 1, 3, 4 — і почав з 5. Keyset продовжив рівно з `id > 3`. Якби новину **додали**, `OFFSET` показав би одну новину двічі.
</details>

### Приклад 5.3. Курсор з двох полів

Задача: стрічку сортують за часом збору (`scraped_at`), а не за `id`. Кілька новин мають однаковий час. Курсор лише за часом (`scraped_at > ?`) пропустить «сусідів» з тим самим часом. Невелика зміна — курсор з пари `(scraped_at, id)` і порівняння пар `(scraped_at, id) > (?, ?)`, яке вміють і SQLite, і PostgreSQL.

In [ ]:
STREAM = [(1, "08:00"), (2, "08:00"), (3, "08:00"), (4, "08:05"), (5, "08:05"), (6, "08:10")]
BY_TIME = "SELECT id, scraped_at FROM stream WHERE scraped_at > ? ORDER BY scraped_at, id LIMIT ?"
BY_PAIR = "SELECT id, scraped_at FROM stream WHERE (scraped_at, id) > (?, ?) ORDER BY scraped_at, id LIMIT ?"

In [ ]:
stream = sqlite3.connect(":memory:")
stream.execute("CREATE TABLE stream (id INTEGER PRIMARY KEY, scraped_at TEXT)")
stream.executemany("INSERT INTO stream VALUES (?, ?)", STREAM)
page1 = stream.execute("SELECT id, scraped_at FROM stream ORDER BY scraped_at, id LIMIT 2").fetchall()
last_id, last_time = page1[-1]
print("сторінка 1:          ", page1)
print("далі за часом:       ", stream.execute(BY_TIME, (last_time, 2)).fetchall())
print("далі за (час, id):   ", stream.execute(BY_PAIR, (last_time, last_id, 2)).fetchall())

За часом наступна сторінка починається з 08:05 — новина 3 (теж 08:00) загубилась. Пара `(час, id)` унікальна, тож продовжує точно після новини 2. Для такого запиту потрібен індекс на обидва стовпці: `(scraped_at, id)`.

### Задача 2. Keyset-пагінація генератором

Таблиця нижче — 100 новин, з яких видалено кожну сьому («дірки» в `id`, як у справжній базі). Напиши генератор `pages(conn, limit)`: кожен `yield` — список `id` однієї сторінки (`SELECT id FROM news WHERE id > ? ORDER BY id LIMIT ?`), курсор — `id` останнього рядка попередньої сторінки; зупинись, коли сторінка коротша за `limit`. Порожню сторінку не віддавай.

In [ ]:
small = sqlite3.connect(":memory:")
small.execute("CREATE TABLE news (id INTEGER PRIMARY KEY, title TEXT)")
small.executemany("INSERT INTO news (id, title) VALUES (?, ?)", ((i, f"n{i}") for i in range(1, 101)))
small.execute("DELETE FROM news WHERE id % 7 = 0")                  # «дірки» в id
print("рядків:", small.execute("SELECT count(*) FROM news").fetchone()[0])

In [ ]:
def pages(conn: sqlite3.Connection, limit: int):
    # YOUR CODE HERE
    pass


In [ ]:
gen = pages(small, 10)
assert gen is not None, "pages має бути генератором: кожну сторінку віддавай через yield"
result = list(gen)
flat = [i for page in result for i in page]
print(len(result), "сторінок; перша:", result[:1], "; остання:", result[-1:])
assert result and all(isinstance(page, list) for page in result), "кожен yield — список id однієї сторінки"
assert flat == [row[0] for row in small.execute("SELECT id FROM news ORDER BY id")], "зібрані id мають збігатися з таблицею"
assert len(flat) == len(set(flat)) == 86, "є повтори: курсор — id останнього рядка, умова id > курсор"
assert list(pages(small, 86))[-1][-1] == 100 and [] not in list(pages(small, 43))
print("✅ Задача 2 розв'язана")

🔮 **Знайди помилку:** студент написав `WHERE id >= ?`. Що побачить клієнт і що буде при `limit=1`?

<details>
<summary>Відповідь</summary>

Кожна сторінка починається з останньої новини попередньої: курсор — `id` уже показаного рядка, його треба виключити (`>`). З `limit=1` сторінка щоразу та сама — нескінченний цикл. Саме тому в перевірці — `len(flat) == len(set(flat))`: повтор ловиться одразу.
</details>

---
## 6. N+1: скільки запитів робить сторінка

`query_count.py` уроку відкриває кожну сторінку проєкту нотаток при 10 і 40 записах і рахує SQL-запити. Йому потрібні Django і весь проєкт, тож запускається він з папки проєкту:

```bash
cd module_5/lessons/lesson_51_ci_cd/crispy_notes_project
python manage.py shell < ../../../../module_6/lessons/lesson_52_final_project/query_count.py
```

Вивід справжнього запуску (з книги):

```text
/notes/        N=10: 8 SQL (HTTP 200)  N=40: 8 SQL (HTTP 200)
/notebooks/    N=10: 7 SQL (HTTP 200)  N=40: 7 SQL (HTTP 200)
/api/notes/    N=10: 4 SQL (HTTP 200)  N=40: 4 SQL (HTTP 200)
```

Кількість запитів не залежить від кількості записів — N+1 немає. Тут відтворимо той самий вимір без Django.

### Приклад 6.1. Лічильник запитів

`sqlite3` вміє викликати функцію на кожен виконаний SQL — `set_trace_callback`. Обгорнемо це в контекстний менеджер: `with QueryCounter(conn) as q:` … `len(q.queries)`.

In [ ]:
class QueryCounter:
    """Збирає всі SQL, виконані з'єднанням усередині with."""

    def __init__(self, conn: sqlite3.Connection) -> None:
        self.conn = conn
        self.queries: list[str] = []

    def __enter__(self) -> "QueryCounter":
        self.conn.set_trace_callback(self.queries.append)
        return self

    def __exit__(self, *exc) -> None:
        self.conn.set_trace_callback(None)

In [ ]:
with QueryCounter(small) as q:
    small.execute("SELECT count(*) FROM news").fetchone()
    for news_id in (1, 2, 3):
        small.execute("SELECT title FROM news WHERE id = ?", (news_id,)).fetchone()
print(len(q.queries), "запити:")
print("\n".join(q.queries))

Лічильник бачить запити вже з підставленими параметрами — зручно, щоб знайти, який саме рядок коду їх робить.

### Приклад 6.2. Сторінка нотаток: N+1 і дві альтернативи

Задача: список нотаток показує назву блокнота й теги кожної нотатки. «Наївна» версія для кожної нотатки робить ще два запити — це і є **N+1** (тут навіть 2N+1). Версія як у `selectors.py` (урок 45) бере блокноти через `JOIN` (`select_related`) і всі теги одним запитом `IN (…)` (`prefetch_related`).

In [ ]:
NOTES_SCHEMA = """
CREATE TABLE notebook (id INTEGER PRIMARY KEY, title TEXT);
CREATE TABLE note (id INTEGER PRIMARY KEY, title TEXT, notebook_id INTEGER REFERENCES notebook(id));
CREATE TABLE note_tag (note_id INTEGER REFERENCES note(id), tag TEXT);
"""


def fill_notes(n: int) -> sqlite3.Connection:
    db = sqlite3.connect(":memory:")
    db.executescript(NOTES_SCHEMA)
    for i in range(n):
        db.execute("INSERT INTO notebook (id, title) VALUES (?, ?)", (i, f"Блокнот {i}"))
        db.execute("INSERT INTO note (id, title, notebook_id) VALUES (?, ?, ?)", (i, f"Нотатка {i}", i))
        db.executemany("INSERT INTO note_tag VALUES (?, ?)", ((i, f"t{k}") for k in range(3)))
    return db


def notes_page_naive(db: sqlite3.Connection) -> list[tuple]:
    rows = []
    for note_id, title, notebook_id in db.execute("SELECT id, title, notebook_id FROM note").fetchall():
        notebook = db.execute("SELECT title FROM notebook WHERE id = ?", (notebook_id,)).fetchone()[0]
        tags = [r[0] for r in db.execute("SELECT tag FROM note_tag WHERE note_id = ?", (note_id,))]
        rows.append((title, notebook, tags))
    return rows


def notes_page_prefetch(db: sqlite3.Connection) -> list[tuple]:
    notes = db.execute("SELECT note.id, note.title, notebook.title FROM note "
                       "JOIN notebook ON notebook.id = note.notebook_id").fetchall()
    marks = ",".join("?" * len(notes))
    tags: dict[int, list[str]] = {}
    for note_id, tag in db.execute(f"SELECT note_id, tag FROM note_tag WHERE note_id IN ({marks})",
                                   [n[0] for n in notes]):
        tags.setdefault(note_id, []).append(tag)
    return [(title, notebook, tags.get(note_id, [])) for note_id, title, notebook in notes]

In [ ]:
for page in (notes_page_naive, notes_page_prefetch):
    report = []
    for n in (10, 40):
        db = fill_notes(n)
        with QueryCounter(db) as q:
            rows = page(db)
        report.append(f"N={n}: {len(q.queries):>2} SQL")
    print(f"{page.__name__:<20} " + "  ".join(report), "| перший рядок:", rows[0])
assert notes_page_naive(fill_notes(5)) == notes_page_prefetch(fill_notes(5))

Результат однаковий (перевіряє `assert`), а запитів — 21 і 81 проти 2 і 2. У наївній версії запитів більшає разом із записами: на сторінці з 1000 нотаток — 2001 похід у базу. **Правило:** кількість запитів сторінки не має залежати від кількості записів, і це варто закріпити тестом (`assertNumQueries` у Django).

### Приклад 6.3. Той самий вимір на SQLAlchemy

Задача: у FastAPI-проєкті (`news_hub`) ORM — SQLAlchemy. Лічильник там — подія `before_cursor_execute` рушія. Невелика зміна — лінивий `relationship` дає N+1, а `selectinload` — рівно два запити.

In [ ]:
from sqlalchemy import ForeignKey, create_engine, event, select
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column, relationship, selectinload


class Base(DeclarativeBase):
    pass


class NotebookRow(Base):
    __tablename__ = "notebooks"
    id: Mapped[int] = mapped_column(primary_key=True)
    title: Mapped[str]


class NoteRow(Base):
    __tablename__ = "notes"
    id: Mapped[int] = mapped_column(primary_key=True)
    title: Mapped[str]
    notebook_id: Mapped[int] = mapped_column(ForeignKey("notebooks.id"))
    notebook: Mapped[NotebookRow] = relationship()          # типово — ліниве завантаження


def count_sql(engine, work) -> int:
    """Скільки SQL виконав рушій під час work()."""
    statements = []

    def listener(conn, cursor, statement, parameters, context, executemany):
        statements.append(statement)

    event.listen(engine, "before_cursor_execute", listener)
    try:
        work()
    finally:
        event.remove(engine, "before_cursor_execute", listener)
    return len(statements)

In [ ]:
engine = create_engine("sqlite://")
Base.metadata.create_all(engine)
with Session(engine) as session:
    session.add_all(NoteRow(title=f"Нотатка {i}", notebook=NotebookRow(title=f"Блокнот {i}")) for i in range(40))
    session.commit()


def lazy():
    with Session(engine) as s:
        [note.notebook.title for note in s.scalars(select(NoteRow))]


def eager():
    with Session(engine) as s:
        [note.notebook.title for note in s.scalars(select(NoteRow).options(selectinload(NoteRow.notebook)))]


print("ліниво:      ", count_sql(engine, lazy), "SQL")
print("selectinload:", count_sql(engine, eager), "SQL")
engine.dispose()

41 запит проти 2: перший — список нотаток, далі по одному на блокнот кожної нотатки. `selectinload` робить другий запит `… WHERE notebooks.id IN (…)` — точно як `prefetch_related` у Django.

---
## 7. Стан і репліки

Throttle DRF у проєкті нотаток — 5 спроб входу за хвилину — рахує спроби в кеші Django. Типовий кеш — пам'ять процесу. Змоделюємо nginx, що по черзі роздає 12 спроб входу між репліками, у кожної з яких свій або спільний кеш.

### Приклад 7.1. Throttle у пам'яті процесу і в Redis

`attempts(replicas, shared)` повертає коди відповідей: 401 — пароль невірний (спробу пропущено), 429 — ліміт. `shared=True` — один словник-кеш на всіх (як Redis), `False` — у кожної репліки свій (пам'ять процесу). Запит `i` іде на репліку `i % replicas` — так nginx роздає по колу.

In [ ]:
RATE = 5


def allow(cache: dict[str, int], ip: str) -> bool:
    """Лічильник спроб з одного IP: пропускаємо перші RATE."""
    cache[ip] = cache.get(ip, 0) + 1
    return cache[ip] <= RATE


def attempts(replicas: int, shared: bool, n: int = 12, ip: str = "172.19.0.1") -> list[int]:
    redis: dict[str, int] = {}
    caches = [redis] * replicas if shared else [{} for _ in range(replicas)]
    return [401 if allow(caches[i % replicas], ip) else 429 for i in range(n)]


def summary(codes: list[int]) -> str:
    return ", ".join(f"{code}×{codes.count(code)}" for code in sorted(set(codes)))

In [ ]:
for replicas in (1, 2, 3):
    print(f"реплік {replicas}: пам'ять процесу → {summary(attempts(replicas, False)):<16} Redis → {summary(attempts(replicas, True))}")
assert summary(attempts(2, False)) == "401×10, 429×2"
assert summary(attempts(2, True)) == summary(attempts(1, False)) == "401×5, 429×7"

Модель збігається зі справжнім стеком уроку 50 (nginx, дві репліки daphne, 12 запитів `POST /api/token/`): з кешем у пам'яті — `401×10, 429×2`, з `CACHES = RedisCache` — `401×5, 429×7`. З трьома репліками в пам'яті процесу ліміт у 5 спроб на 12 запитах не спрацьовує взагалі. Звичайні тести цього не бачать: вони запускають один процес.

### Приклад 7.2. Кеш стрічки: застарілі дані на іншій репліці

Задача: та сама проблема з кешем даних. `news_hub` кешує стрічку під ключем з версією (`news:v{версія}:list`) і після зміни збільшує `news:version`. Невелика зміна до 7.1 — дві репліки, одна додає новину; що віддасть друга?

In [ ]:
class Replica:
    """Репліка API: кеш стрічки під ключем з версією (як NewsCache у news_hub)."""

    def __init__(self, name: str, store: dict) -> None:
        self.name, self.store = name, store

    def feed(self, db: list[str]) -> list[str]:
        key = f"news:v{self.store.get('news:version', 0)}:list"
        if key not in self.store:
            self.store[key] = list(db)                     # «запит до бази» і збереження в кеш
        return self.store[key]

    def add_news(self, db: list[str], title: str) -> None:
        db.append(title)
        self.store["news:version"] = self.store.get("news:version", 0) + 1     # інвалідація після COMMIT

In [ ]:
for shared in (False, True):
    db = ["Новина 1", "Новина 2"]
    redis: dict = {}
    a = Replica("api-1", redis if shared else {})
    b = Replica("api-2", redis if shared else {})
    a.feed(db), b.feed(db)                                 # обидві закешували стрічку
    a.add_news(db, "Новина 3")
    where = "Redis (спільний)" if shared else "пам'ять процесу"
    print(f"{where:<16} api-1: {a.feed(db)}  api-2: {b.feed(db)}")

З кешем у пам'яті `api-2` не знає про нову версію і ще довго віддає стару стрічку: користувач оновлює сторінку й бачить то 3 новини, то 2 — залежно від того, куди nginx відправив запит. Зі спільним сховищем версія одна для всіх.

**Правило для фінального проєкту:** усе, що має бути спільним (лічильники, кеш, сесії, черги), — у базі чи Redis; у пам'яті процесу — лише те, що можна втратити й порахувати заново.

---
## 8. Чек-лист репозиторію

Останній крок аудиту — сам репозиторій: чи запуститься проєкт на чужій машині, і чи не потрапило в Git те, чого там бути не має.

### Приклад 8.1. Compose як дані

Задача: перевірити `docker-compose.yml`, не запускаючи Docker. Це YAML — розбираємо `yaml.safe_load` (розуміє й `<<: *app` — злиття якорів) і для кожного сервісу збираємо: чи є healthcheck, на кого чекає і які порти відкриває назовні.

In [ ]:
import yaml


def compose_report(path: Path) -> list[tuple[str, str, str, str]]:
    """(сервіс, healthcheck, depends_on з умовами, порти назовні) для кожного сервісу Compose."""
    spec = yaml.safe_load(path.read_text(encoding="utf-8"))
    rows = []
    for name, service in spec["services"].items():
        health = service.get("healthcheck")
        health_text = "—" if health is None else "вимкнено" if health.get("disable") else "є"
        deps = service.get("depends_on", {})
        deps_text = ", ".join(f"{d}:{c.get('condition', '').removeprefix('service_')}" for d, c in deps.items()) \
            if isinstance(deps, dict) else ", ".join(deps)
        rows.append((name, health_text, deps_text or "—", ", ".join(map(str, service.get("ports", []))) or "—"))
    return rows

In [ ]:
for project in ("news_hub", "crispy_notes_project"):
    print(f"── {project}")
    print(f"   {'сервіс':<10} {'health':<9} {'чекає на':<46} порти")
    for name, health, deps, ports in compose_report(ROOT / PROJECTS / project / "docker-compose.yml"):
        print(f"   {name:<10} {health:<9} {deps:<46} {ports}")

Читаємо так само, як у уроці 50: база й Redis — з healthcheck і **без портів назовні**; міграції (`migrate` / `release`) — разовий сервіс, на завершення якого (`completed_successfully`) чекає API; назовні відкритий лише nginx (і двійник Telegram — лише на `127.0.0.1`). Healthcheck «вимкнено» у міграцій, бота й двійника — свідомо: міграції й бот не слухають порт, двійник потрібен лише на занятті.

### Приклад 8.2. `docker compose config` — якщо є Docker

Задача: остаточно перевірити файл може лише сам Compose: підстановку змінних, якорі, обов'язкові значення `${VAR:?…}`. У Colab Docker-демона немає — клітинка це перевіряє і пропускає себе з поясненням. Якщо Docker є, копіюємо `docker-compose.yml` і `.env.example` у тимчасову папку (справжній `.env` не потрібен) і запускаємо `config` двічі: з порожніми значеннями з прикладу і з заповненими. `config` нічого не запускає і не створює.

In [ ]:
def docker_ready() -> bool:
    """Є команда docker і працює демон (docker info)."""
    if shutil.which("docker") is None:
        return False
    return subprocess.run(["docker", "info"], capture_output=True).returncode == 0


def compose_config(project: str, fill: bool) -> str:
    """docker compose config у копії проєкту; .env — з .env.example (порожні значення заповнюємо, якщо fill)."""
    target = WORK / f"compose_{project}"
    target.mkdir(exist_ok=True)
    shutil.copy(ROOT / PROJECTS / project / "docker-compose.yml", target)
    lines = (ROOT / PROJECTS / project / ".env.example").read_text(encoding="utf-8").splitlines()
    env = [line + "notebook-check" if fill and line.endswith("=") and not line.startswith("#") else line for line in lines]
    (target / ".env").write_text("\n".join(env) + "\n", encoding="utf-8")
    run = subprocess.run(["docker", "compose", "config", "--services"], cwd=target, capture_output=True, text=True)
    return ("OK: " + " ".join(run.stdout.split())) if run.returncode == 0 else "помилка: " + run.stderr.strip()[-160:]

In [ ]:
if docker_ready():
    for project in ("news_hub", "crispy_notes_project"):
        print(f"{project}, порожній .env.example:  {compose_config(project, fill=False)}")
        print(f"{project}, заповнений:            {compose_config(project, fill=True)}")
else:
    print("Docker недоступний (у Colab його немає) — пропускаємо. Офлайн-перевірка — приклад 8.1.")

Вивід справжнього запуску (Docker 29):

```text
news_hub, порожній .env.example:  помилка: … required variable POSTGRES_PASSWORD is missing a value: задай POSTGRES_PASSWORD у .env
news_hub, заповнений:            OK: postgres migrate redis api nginx
crispy_notes_project, порожній .env.example:  помилка: … required variable POSTGRES_PASSWORD is missing a value: …
crispy_notes_project, заповнений:            OK: redis db release web nginx
```

Помилка з порожнім паролем — **бажана**: `${POSTGRES_PASSWORD:?…}` у Compose не дає запустити стек без пароля бази. `bot` і `twin` у списку немає — вони в профілях (`--profile polling`, `--profile twin`).

### Приклад 8.3. Залежності: що зафіксовано

Задача: чи збереться проєкт через пів року? `requirements.txt` — текст, рядок за рядком: назва, умова версії, коментар. Порахуємо, скільки залежностей мають нижню межу (`>=`), верхню (`<`) і скільки взагалі без версії.

In [ ]:
import re

REQUIREMENT = re.compile(r"^\s*([A-Za-z0-9_.\-]+(?:\[[^\]]+\])?)\s*([^#\s]*)\s*(?:#\s*(.*))?$")


def parse_requirements(path: Path) -> list[tuple[str, str, str]]:
    """(пакет, умова версії, коментар) для кожного рядка-залежності requirements.txt."""
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.strip() and not line.lstrip().startswith("#"):
            name, spec, note = REQUIREMENT.match(line).groups()
            rows.append((name, spec, note or ""))
    return rows

In [ ]:
for project in ("news_hub", "crispy_notes_project"):
    reqs = parse_requirements(ROOT / PROJECTS / project / "requirements.txt")
    lower = sum(">=" in spec for _, spec, _ in reqs)
    upper = sum("<" in spec.replace("<=", "<") for _, spec, _ in reqs)
    bare = [name for name, spec, _ in reqs if not spec]
    noted = sum(bool(note) for *_, note in reqs)
    print(f"{project:<22} {len(reqs):>2} пакетів: з нижньою межею {lower}, з верхньою {upper}, "
          f"без версії {bare or 0}; з поясненням {noted}")
print("приклад рядка:", parse_requirements(ROOT / PROJECTS / "news_hub" / "requirements.txt")[-1])

В обох проєктах кожна залежність має нижню межу, і більшість — з коментарем «чому саме ця версія». Нижні межі не вигадані: CI уроку 51 (job `min-versions`) ставить саме мінімальні версії й ганяє тести. Верхня межа лише в Django (`<6.0`) — мажорна версія фреймворку ламає сумісність.

### Приклад 8.4. Секрети в коді

Задача: знайти в коді рядки, схожі на секрети: `SECRET…`, `PASSWORD…`, `TOKEN…`, `API_KEY…`, яким присвоєно рядок довжиною від 6 символів. Тести пропускаємо — там фіктивні паролі доречні. Скрипт знаходить **кандидатів**; вирішує людина.

In [ ]:
SECRET = re.compile(r"""(secret|password|token|api_key)\w*\s*[:=]\s*["'][^"']{6,}["']""", re.I)


def find_secrets(packages: list[Path]) -> list[str]:
    """'файл:рядок: текст' для кожного рядка-кандидата в пакетах (без тестів)."""
    hits = []
    for package in packages:
        for path in sorted(package.rglob("*.py")):
            if "tests" in path.parts or path.name.startswith("test"):
                continue
            for number, line in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):
                if SECRET.search(line):
                    hits.append(f"{path.relative_to(package.parent)}:{number}: {line.strip()[:70]}")
    return hits

In [ ]:
for hit in find_secrets([NEWS, NOTES / "hello_app", NOTES / "hello_project"]):
    print(hit)

Два кандидати, і обидва — не витік:

- `TELEGRAM_SECRET_HEADER` — **назва** заголовка HTTP, а не значення секрету;
- `DEV_SECRET_KEY` — ключ лише для навчального запуску з `DEBUG=1`. У `settings.py` поруч стоїть захист: з `DJANGO_DEBUG=0` і цим ключем Django **не стартує** (`ImproperlyConfigured`) — краще не працювати зовсім, ніж працювати з ключем, який є в Git.

У фінальному проєкті справжні секрети — лише зі змінних середовища, `.env` — у `.gitignore`, а в репозиторії — `.env.example` з порожніми значеннями.

### Задача 3. Чек-лист фінального проєкту

Напиши `check_project(root)` → словник `{пункт: True/False}` рівно з шістьма пунктами:

- `README` — є `README.md`;
- `залежності` — `requirements.txt` або `pyproject.toml`;
- `тести` — будь-де файл `test*.py`;
- `міграції` — будь-де папка `migrations`;
- `.env.example` — є, а `.env` згадано в `.gitignore` окремим рядком (сам файл `.env` у репозиторій не потрапляє);
- `Docker` — є `Dockerfile` і `docker-compose.yml`.

Перевірка: обидва проєкти курсу проходять увесь чек-лист, а проєкт першого дня (лише `main.py`) — жодного пункту.

In [ ]:
def check_project(root: Path) -> dict[str, bool]:
    # YOUR CODE HERE
    pass


In [ ]:
def show(name: str, report: dict[str, bool]) -> None:
    print(f"{name:<22}", "  ".join(("✅ " if ok else "❌ ") + item for item, ok in report.items()))


first_day = WORK / "my_final_project"
first_day.mkdir(exist_ok=True)
(first_day / "main.py").write_text("print('hello')\n", encoding="utf-8")
for name in ("news_hub", "crispy_notes_project"):
    report = check_project(ROOT / PROJECTS / name)
    assert isinstance(report, dict) and len(report) == 6, "словник рівно з шістьма пунктами"
    show(name, report)
    assert all(report.values()), f"{name}: проєкт курсу має пройти всі пункти — перевір, який False"
show("my_final_project", check_project(first_day))
assert not any(check_project(first_day).values()), "у проєкті з одним main.py жоден пункт не виконаний"
print("✅ Задача 3 розв'язана")

Обидва проєкти курсу проходять увесь чек-лист. Порожній проєкт — ні в чому, і це нормально для першого дня. План з книги: скелет з README, першим тестом і CI — **до** першої функції, а Docker і Compose — щойно з'явиться база. Запусти `check_project(Path("шлях/до/проєкту"))` на своєму проєкті. CI (`.github/workflows/`) тут не перевіряємо: у курсі workflows лежать у корені репозиторію, а не в папці проєкту.

### Прибираємо

Тимчасова папка з тестовими пакетами, копією `news_hub` і проєктом першого дня більше не потрібна; з'єднання з базами в пам'яті закриваємо.

In [ ]:
for db in (conn, search, small, stream):
    db.close()
shutil.rmtree(WORK, ignore_errors=True)
print("тимчасова папка прибрана:", not WORK.exists())

---
## Самоперевірка

1. Що означає високий fan-in і нульовий fan-out модуля?
2. Чому в циклі після ребра `db → repository` опинився й `tables`?
3. Чому відкладений імпорт у функції не прибирає цикл з аудиту?
4. Чому індекс на `lower(category)` не допоміг запиту `WHERE category = ?`?
5. Чим keyset-пагінація краща за OFFSET, крім швидкості?
6. Як помітити N+1, не читаючи код?
7. Чому помилку з throttle у пам'яті процесу не ловлять звичайні тести?

<details>
<summary>Відповіді</summary>

1. Фундамент: від модуля залежить багато інших, а він — ні від кого. Його інтерфейс змінювати найдорожче, тож тестів у нього має бути найбільше.
2. `repository → tables → db → repository`: `tables` лежить на шляху між двома модулями циклу, отже, теж залежить сам від себе.
3. Залежність лишилась — її лише відклали до виклику. Модулі досі не можна зрозуміти й змінити окремо один від одного.
4. Індекс зберігає значення `lower(category)`; вираз `category` для бази інший.
5. Сторінки не зсуваються, коли між запитами додають чи видаляють рядки; немає повторів і пропусків.
6. Порахувати запити сторінки при 10 і 40 записах: якщо число росте разом із записами — у циклі є запит.
7. Тести запускають один процес — один кеш. Лічильники розходяться лише з кількома репліками.
</details>

## Далі

- Книга курсу: [Урок 52](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m6/lesson_52/) — зріз курсу, схеми баз обох проєктів, `EXPLAIN ANALYZE` на PostgreSQL (trigram, keyset, зайві індекси), граф залежностей, стан і репліки, вимоги та план фінального проєкту.
- Інструменти уроку: `architecture_audit.py` (+ `test_architecture_audit.py`), `db_audit.sql`, `query_count.py`, `patches/news_hub_search_index.patch`.
- Урок 53 — презентація фінального проєкту.